# 🚲 Bicycle RTI Hackathon — One-Click Deployment

This notebook deploys the **complete** Bicycle Real-Time Intelligence solution into your workspace.

## What Gets Deployed (23 items)
| Stage | Items |
|-------|-------|
| 1. Storage | 4 Lakehouses |
| 2. Real-Time | 1 Eventhouse → 1 KQL Database (depends on Eventhouse) |
| 3. Compute & Ingestion | 9 Notebooks, 2 Eventstreams |
| 4. Analytics | 2 Semantic Models, 1 Pipeline |
| 5. Presentation | 1 KQL Dashboard, 2 Data Agents |

## Before you start (one-time setup)

1. Create a **temporary Lakehouse** in your workspace (e.g. name it `deploy_staging`)
2. Open the lakehouse → click **Upload → Upload folder**
3. Upload the **`workspace`** folder from the cloned repo
4. Upload the **`sample_data`** folder from the cloned repo (for dim_customers table)
5. You should now see `Files/workspace/` and `Files/sample_data/` in the lakehouse
6. **Attach** this lakehouse to this notebook (left sidebar → "Add lakehouse" → select `deploy_staging`)

## Instructions

1. Run **Cell 1** — installs `fabric-cicd` (one-time)
2. Run **Cell 2** — deploys all 23 items from the lakehouse files (no GitHub, no PAT, no auth prompts)
3. Run **Cell 2.5** — uploads sample_data to bicycles_gold lakehouse
4. Run **Cell 3** — auto-fixes KQL Dashboard URI + Pipeline placeholders
5. Run **Cell 4** — validates deployment
6. Run **Cell 5** — organizes items into workspace folders
7. See **Cell 6** — remaining manual steps
8. **(Optional)** Delete the `deploy_staging` lakehouse after deployment

In [ ]:
# =============================================================
# CELL 1 — Install dependencies
# =============================================================
# Pip dependency warnings from nni/mlflow/datasets are harmless —
# they are pre-installed Fabric runtime packages, not ours.

%pip install fabric-cicd --quiet

# Restart Python so the newly installed package is importable
try:
    import notebookutils
    notebookutils.session.restartPython()
except NameError:
    print("⚠️  notebookutils not available — restart the kernel manually before Cell 2.")

In [ ]:
# =============================================================
# CELL 2 — Deploy all 21 items from Lakehouse
# =============================================================
import os, shutil, tempfile, json, base64, time, requests, sys, traceback
import sempy.fabric as fabric
from fabric_cicd import FabricWorkspace, publish_all_items

# ── Monkey-patch fabric-cicd bug: item_metadata UnboundLocalError ──
# fabric-cicd creates ParsingError but doesn't raise it, so if a .platform
# file read fails, item_metadata stays unbound and crashes. Fix: skip + warn.
def _patch_fabric_cicd():
    import fabric_cicd.fabric_workspace as _fw
    import inspect, textwrap, logging
    _logger = logging.getLogger("fabric_cicd")
    try:
        src = inspect.getsource(_fw.FabricWorkspace._refresh_repository_items)
    except Exception:
        return  # Can't inspect — skip
    # Only patch if the bug is present (ParsingError without raise)
    if "ParsingError(msg, logger)" in src and "raise ParsingError(msg, logger)" not in src:
        original = _fw.FabricWorkspace._refresh_repository_items
        def patched_refresh(self):
            """Patched: skip items with unreadable .platform instead of crashing."""
            import json as _json
            from pathlib import Path
            self.repository_items = {}
            empty_logical_id_paths = []
            visited_logical_ids = set()
            for root, _dirs, files in os.walk(self.repository_directory):
                directory = Path(root)
                if ".platform" not in files:
                    continue
                item_metadata_path = directory / ".platform"
                if not any(directory.iterdir()):
                    continue
                try:
                    with open(item_metadata_path, encoding="utf-8") as f:
                        item_metadata = _json.load(f)
                except (FileNotFoundError, _json.JSONDecodeError) as e:
                    _logger.warning(f"Skipping {directory.name}: {e}")
                    continue
                md = item_metadata.get("metadata", {})
                if "type" not in md or "displayName" not in md:
                    _logger.warning(f"Skipping {directory.name}: missing type/displayName")
                    continue
                item_type = md["type"]
                item_name = md["displayName"]
                item_description = md.get("description", "")
                item_logical_id = item_metadata.get("config", {}).get("logicalId", "")
                if not item_logical_id or item_logical_id.strip() == "":
                    empty_logical_id_paths.append(str(item_metadata_path))
                    continue
                item_path = directory
                relative_path = f"/{directory.relative_to(self.repository_directory).as_posix()}"
                self.repository_items[item_name] = _fw.Item(
                    item_name=item_name,
                    item_type=item_type,
                    item_path=item_path,
                    item_logical_id=item_logical_id,
                    item_description=item_description,
                    item_relative_path=relative_path,
                )
            if empty_logical_id_paths:
                _logger.warning(f"Empty logicalId in: {empty_logical_id_paths}")
        _fw.FabricWorkspace._refresh_repository_items = patched_refresh
        print("   🔧 Patched fabric-cicd item_metadata bug")
    else:
        print("   ✅ fabric-cicd already has the fix")

_patch_fabric_cicd()


# ── Source workspace GUID (baked into exported files) ──
# This must match the workspace these items were originally exported from.
# The deploy process replaces it with the TARGET workspace ID at deploy time.
SOURCE_WORKSPACE_ID = "573cc7c7-a45a-4fd9-886e-9db4e9798db4"
ALERT_PLACEHOLDER = "__ALERT_RECIPIENT_EMAIL__"

# ── Locate workspace/ folder in the attached lakehouse ──
LAKEHOUSE_PATH = "/lakehouse/default/Files/workspace"

if not os.path.isdir(LAKEHOUSE_PATH):
    alt_paths = [
        "/lakehouse/default/Files/RTI-Hackathon-Demo/workspace",
        "/lakehouse/default/Files/RTI-Hackathon-Demo-main/workspace",
    ]
    found = None
    for p in alt_paths:
        if os.path.isdir(p):
            found = p
            break
    if found:
        LAKEHOUSE_PATH = found
    else:
        print("❌ Cannot find workspace/ folder in the attached lakehouse.")
        print("   Expected: /lakehouse/default/Files/workspace/")
        print("   Fix: Open your lakehouse → Upload → Upload folder → select the 'workspace' folder")
        raise FileNotFoundError(f"workspace/ not found at {LAKEHOUSE_PATH}")

all_items = [d for d in os.listdir(LAKEHOUSE_PATH)
             if os.path.isdir(os.path.join(LAKEHOUSE_PATH, d))]
print(f"📂 Found {len(all_items)} items in {LAKEHOUSE_PATH}")

# ── Build item-type index ──
item_index = {}
for folder_name in all_items:
    parts = folder_name.rsplit(".", 1)
    if len(parts) == 2:
        item_index.setdefault(parts[1], []).append(folder_name)
print("   Item types found:", {k: len(v) for k, v in item_index.items()})

ws_id = fabric.get_workspace_id()
print(f"\n🚀 Deploying to workspace: {ws_id}")

if ws_id == SOURCE_WORKSPACE_ID:
    print("   ℹ️  Target = source workspace — no GUID replacement needed")
else:
    print(f"   🔄 Will replace source workspace GUID {SOURCE_WORKSPACE_ID}")
    print(f"      with target workspace GUID {ws_id}")

# ── Auto-detect deploying user's email for Activator alerts ──
def _get_user_email():
    """Extract UPN from Fabric auth token (JWT)."""
    try:
        tok = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
        payload = tok.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        claims = json.loads(base64.b64decode(payload))
        return claims.get("upn") or claims.get("unique_name") or claims.get("preferred_username", "")
    except Exception:
        return ""

ALERT_EMAIL = _get_user_email()
if ALERT_EMAIL:
    print(f"   📧 Alert email (from token): {ALERT_EMAIL}")
else:
    print("   ⚠️ Could not detect email — Activator alerts will keep placeholder")

# ── Helper: get access token ──
def get_token():
    try:
        return notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
    except Exception:
        try:
            return notebookutils.credentials.getToken("pbi")
        except Exception:
            from notebookutils import credentials
            return credentials.getToken("https://api.fabric.microsoft.com")

def get_kusto_token():
    """Get token scoped for Kusto management endpoint."""
    for scope in ["https://kusto.kusto.windows.net", "https://api.fabric.microsoft.com", "pbi"]:
        try:
            return notebookutils.credentials.getToken(scope)
        except Exception:
            continue
    raise RuntimeError("Could not get Kusto token")

# ── Helper: replace source workspace GUID with target in staging dir ──
TEXT_EXTENSIONS = {".json", ".tmdl", ".pbism", ".ipynb", ".xml", ".yml", ".yaml", ".md"}

def patch_workspace_guid(stage_ws):
    """Replace SOURCE_WORKSPACE_ID with ws_id in all text files under stage_ws."""
    if ws_id == SOURCE_WORKSPACE_ID:
        return 0  # Same workspace, nothing to patch
    count = 0
    for root, dirs, files in os.walk(stage_ws):
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            if ext not in TEXT_EXTENSIONS and fname != ".platform":
                continue
            fpath = os.path.join(root, fname)
            try:
                with open(fpath, "r", encoding="utf-8") as f:
                    content = f.read()
                if SOURCE_WORKSPACE_ID in content:
                    content = content.replace(SOURCE_WORKSPACE_ID, ws_id)
                    with open(fpath, "w", encoding="utf-8") as f:
                        f.write(content)
                    count += 1
            except (UnicodeDecodeError, PermissionError):
                continue
    if count:
        print(f"   🔄 Patched workspace GUID in {count} file(s)")
    return count

def patch_alert_email(stage_ws):
    """Replace __ALERT_RECIPIENT_EMAIL__ with ALERT_EMAIL in all text files."""
    if not ALERT_EMAIL:
        return 0
    count = 0
    for root, dirs, files in os.walk(stage_ws):
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            if ext not in TEXT_EXTENSIONS and fname != ".platform":
                continue
            fpath = os.path.join(root, fname)
            try:
                with open(fpath, "r", encoding="utf-8") as f:
                    content = f.read()
                if ALERT_PLACEHOLDER in content:
                    content = content.replace(ALERT_PLACEHOLDER, ALERT_EMAIL)
                    with open(fpath, "w", encoding="utf-8") as f:
                        f.write(content)
                    count += 1
            except (UnicodeDecodeError, PermissionError):
                continue
    if count:
        print(f"   📧 Patched alert email in {count} file(s)")
    return count

# ── Helper: create isolated temp dir for a fabric-cicd stage ──
def make_stage_dir(type_list, ref_types=None):
    """Copy folders for given item types into a temp dir.
    For ref_types, copy only .platform files (for logicalId resolution)."""
    stage_dir = tempfile.mkdtemp(prefix="rti_stage_")
    stage_ws = os.path.join(stage_dir, "workspace")
    os.makedirs(stage_ws)
    folders = []
    for t in type_list:
        folders.extend(item_index.get(t, []))
    for f in folders:
        shutil.copytree(os.path.join(LAKEHOUSE_PATH, f),
                        os.path.join(stage_ws, f))
    # Add reference-only .platform files for logicalId resolution
    if ref_types:
        for t in ref_types:
            for f in item_index.get(t, []):
                if os.path.exists(os.path.join(stage_ws, f)):
                    continue  # Already copied as full folder
                ref_dir = os.path.join(stage_ws, f)
                os.makedirs(ref_dir, exist_ok=True)
                src_platform = os.path.join(LAKEHOUSE_PATH, f, ".platform")
                if os.path.exists(src_platform):
                    shutil.copy2(src_platform, os.path.join(ref_dir, ".platform"))
    # Replace source workspace GUID with target
    patch_workspace_guid(stage_ws)
    # Replace alert email placeholder with deploying user's email
    patch_alert_email(stage_ws)
    return stage_dir, stage_ws, folders

# ── Helper: wait for LRO ──
def wait_lro(resp, headers, label, max_wait=180):
    if resp.status_code in (200, 201):
        return resp.json() if resp.text else True
    if resp.status_code != 202:
        return None
    op_url = resp.headers.get("Location")
    if not op_url:
        time.sleep(15)
        return True
    elapsed = 0
    while elapsed < max_wait:
        retry = int(resp.headers.get("Retry-After", 5))
        time.sleep(retry)
        elapsed += retry
        r = requests.get(op_url, headers=headers)
        if r.status_code == 200:
            status = r.json().get("status", "")
            if status == "Succeeded":
                return r.json()
            if status in ("Failed", "Cancelled"):
                print(f"   ❌ {label}: {status} — {r.json().get('error', {}).get('message', '')}")
                return None
    print(f"   ⚠️ {label}: timed out ({max_wait}s)")
    return None

API = "https://api.fabric.microsoft.com/v1"

stage_status = {}

# ═══════════════════════════════════════════════════════════════
# STAGE 1/5: Lakehouses (fabric-cicd)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("🚀 Stage 1/5: Lakehouses")
print(f"{'='*60}")
try:
    stage_dir, stage_ws, folders = make_stage_dir(["Lakehouse"])
    print(f"   📦 {', '.join(f.rsplit('.', 1)[0] for f in folders)}")
    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=["Lakehouse"])
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    stage_status["1_Lakehouses"] = True
    print("   ✅ Stage 1/5 complete")
except Exception as e:
    stage_status["1_Lakehouses"] = False
    print(f"   ❌ Stage 1/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

try:
    stage_dir, stage_ws, folders = make_stage_dir(["Eventhouse"])
    print(f"   📦 {', '.join(f.rsplit('.', 1)[0] for f in folders)}")
    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=["Eventhouse"])
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    stage_status["2_Eventhouse"] = True
    print("   ✅ Stage 2/5 complete")
except Exception as e:
    stage_status["2_Eventhouse"] = False
    print(f"   ❌ Stage 2/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

# ═══════════════════════════════════════════════════════════════
try:
    token = get_token()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    if not stage_status.get("2_Eventhouse"):
        raise RuntimeError("Skipped — Stage 2 (Eventhouse) did not succeed")

    # Find the deployed Eventhouse
    resp = requests.get(f"{API}/workspaces/{ws_id}/eventhouses", headers=headers)
    resp.raise_for_status()
    eventhouse_id = None
    for eh in resp.json().get("value", []):
      if eh["displayName"] == "bikerentaleventhouse":
          eventhouse_id = eh["id"]
          break
          eventhouse_id = eh["id"]
    if not eventhouse_id:
      raise RuntimeError("❌ Eventhouse 'bikerentaleventhouse' not found — Stage 2 may have failed")
    print(f"   Found Eventhouse: {eventhouse_id}")

    # Check if KQL Database already exists
    resp = requests.get(f"{API}/workspaces/{ws_id}/kqlDatabases", headers=headers)
    resp.raise_for_status()
    kqldb_id = None
    for db in resp.json().get("value", []):
      if db["displayName"] == "bikerentaleventhouse":
          kqldb_id = db["id"]
          break

    if kqldb_id:
      print(f"   ℹ️  KQL Database already exists: {kqldb_id}")
    else:
      # Create KQL Database under the Eventhouse
      print("   Creating KQL Database 'bikerentaleventhouse'...")
      create_body = {
          "displayName": "bikerentaleventhouse",
          "creationPayload": {
              "databaseType": "ReadWrite",
              "parentEventhouseItemId": eventhouse_id,
              "oneLakeCachingPeriod": "P36500D",
              "oneLakeStandardStoragePeriod": "P36500D",
          },
      }
      resp = requests.post(f"{API}/workspaces/{ws_id}/kqlDatabases",
                           headers=headers, json=create_body)
      print(f"   Response: HTTP {resp.status_code}")
      if resp.status_code in (200, 201):
          kqldb_id = resp.json().get("id")
          print(f"   ✅ Created KQL Database: {kqldb_id}")
      elif resp.status_code == 202:
          result = wait_lro(resp, headers, "Create KQLDatabase")
          if result:
              # Re-fetch to get the ID
              resp2 = requests.get(f"{API}/workspaces/{ws_id}/kqlDatabases", headers=headers)
              for db in resp2.json().get("value", []):
                  if db["displayName"] == "bikerentaleventhouse":
                      kqldb_id = db["id"]
                      break
              if kqldb_id:
                  print(f"   ✅ Created KQL Database: {kqldb_id}")
              else:
                  raise RuntimeError("❌ KQL Database created but could not find it")
          else:
              raise RuntimeError("❌ Failed to create KQL Database (LRO failed)")
      else:
          error_detail = resp.text[:800]
          print(f"   Error detail: {error_detail}")
          raise RuntimeError(f"❌ Failed to create KQL Database: HTTP {resp.status_code}")

    # Now run the schema KQL commands via the Kusto management API
    print("   Running schema commands...")
    kql_schema_path = None
    for f in item_index.get("KQLDatabase", []):
      schema_file = os.path.join(LAKEHOUSE_PATH, f, "DatabaseSchema.kql")
      if os.path.exists(schema_file):
          kql_schema_path = schema_file
          break

    if kql_schema_path:
      # Wait for query URI to become available (may take a few seconds after creation)
      query_uri = None
      print("   Waiting for KQL Database query URI...")
      for attempt in range(12):  # Retry up to ~60 seconds
          resp = requests.get(f"{API}/workspaces/{ws_id}/kqlDatabases/{kqldb_id}", headers=headers)
          if resp.status_code == 200:
              props = resp.json().get("properties", {})
              query_uri = props.get("queryUri") or props.get("parentEventhouseUri")
              if query_uri:
                  break
          if attempt < 11:
              time.sleep(5)

      if query_uri:
          print(f"   Query URI: {query_uri}")
          # Read schema commands
          with open(kql_schema_path, "r", encoding="utf-8") as sf:
              schema_text = sf.read()

          # Parse individual commands (split on lines starting with .)
          commands = []
          current = []
          for line in schema_text.split("\n"):
              stripped = line.strip()
              if stripped.startswith(".") and current:
                  commands.append("\n".join(current))
                  current = [line]
              elif stripped and not stripped.startswith("//"):
                  current.append(line)
          if current:
              commands.append("\n".join(current))

          # Execute each management command with Kusto-scoped token
          kusto_headers = {
              "Authorization": f"Bearer {get_kusto_token()}",
              "Content-Type": "application/json",
          }
          success_count = 0
          for cmd in commands:
              cmd_clean = cmd.strip()
              if not cmd_clean or cmd_clean.startswith("//"):
                  continue
              try:
                  mgmt_resp = requests.post(
                      f"{query_uri}/v1/rest/mgmt",
                      headers=kusto_headers,
                      json={"csl": cmd_clean, "db": "bikerentaleventhouse"},
                  )
                  if mgmt_resp.status_code == 200:
                      success_count += 1
                  else:
                      first_line = cmd_clean.split("\n")[0][:60]
                      print(f"   ⚠️ Command failed ({mgmt_resp.status_code}): {first_line}...")
                      print(f"      Detail: {mgmt_resp.text[:200]}")
              except Exception as e:
                  print(f"   ⚠️ Command error: {e}")

          print(f"   ✅ Schema: {success_count}/{len(commands)} commands succeeded")
      else:
          print("   ⚠️ Could not get query URI after 60s — schema will need manual setup")
    else:
      print("   ⚠️ DatabaseSchema.kql not found")

    stage_status["3_KQLDatabase"] = True
    print("   ✅ Stage 3/5 complete")
except Exception as e:
    stage_status["3_KQLDatabase"] = False
    print(f"   ❌ Stage 3/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

# ═══════════════════════════════════════════════════════════════
# STAGE 4/5: Semantic Models + Notebooks (fabric-cicd)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("🚀 Stage 4/5: Semantic Models + Notebooks")
print(f"{'='*60}")
try:
    stage_dir, stage_ws, _ = make_stage_dir(["SemanticModel", "Notebook"],
                                            ref_types=["KQLDatabase", "Lakehouse", "Eventhouse"])
    deploy_types = ["SemanticModel", "Notebook"]
    folders = []
    for t in deploy_types:
        folders.extend(item_index.get(t, []))
    print(f"   📦 Deploying: {', '.join(f.rsplit('.', 1)[0] for f in folders)}")

    # Sanitize pipeline: strip activities with unresolvable externalReferences
    for pl_folder in item_index.get("DataPipeline", []):
        pl_path = os.path.join(stage_ws, pl_folder, "pipeline-content.json")
        if os.path.exists(pl_path):
            with open(pl_path, "r", encoding="utf-8") as pf:
                pl = json.load(pf)
            original_count = len(pl.get("properties", {}).get("activities", []))
            pl["properties"]["activities"] = [
                a for a in pl["properties"]["activities"]
                if not any("__" in str(v) for v in a.get("externalReferences", {}).values())
            ]
            removed = original_count - len(pl["properties"]["activities"])
            if removed:
                print(f"   🔧 Stripped {removed} unresolvable activity(ies) from pipeline")
            with open(pl_path, "w", encoding="utf-8") as pf:
                json.dump(pl, pf, indent=2, ensure_ascii=False)

    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=deploy_types)
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    stage_status["4_SM_Notebooks"] = True
    print("   ✅ Stage 4/5 complete")
except Exception as e:
    stage_status["4_SM_Notebooks"] = False
    print(f"   ❌ Stage 4/5 FAILED: {e}")
    print("   ⚠️ Continuing with remaining stages...")

# ═══════════════════════════════════════════════════════════════
# STAGE 5/5: Eventstreams (REST API) + Pipeline + Dashboard + Agents
# ═══════════════════════════════════════════════════════════════
# Eventstreams contain hardcoded destination GUIDs (logicalIds from source).
# fabric-cicd can't resolve these to the actual item GUIDs in the target
# workspace. So we deploy Eventstreams via REST API after looking up the
# real Lakehouse/KQLDatabase IDs that Stages 1-3 created.
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("🚀 Stage 5/5: Eventstreams + Analytics + Presentation")
print(f"{'='*60}")
sys.stdout.flush()

# --- 5a: Eventstreams via REST API (with resolved destination GUIDs) ---
es_ok = True
es_error_detail = ""  # Captured for final summary (Fabric output can truncate mid-cell)
try:
    print("   🔄 Resolving Eventstream destination GUIDs...")
    sys.stdout.flush()
    token = get_token()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    # Build logicalId → actual GUID mapping for Eventstream destinations
    #   logicalId (from source .platform)    →  displayName
    #   890af2be-6762-5333-8e9f-f44bd626c040 →  bikerental_bronze_raw  (Lakehouse)
    #   97f37696-79a2-52ad-8fd7-c78a8ecd6106 →  weather_bronze_raw     (Lakehouse)
    #   27e1952e-cbd2-5a14-b54b-79f090db2745 →  bikerentaleventhouse   (KQLDatabase)

    LOGICAL_IDS = {
        "890af2be-6762-5333-8e9f-f44bd626c040": ("Lakehouse",    "bikerental_bronze_raw"),
        "97f37696-79a2-52ad-8fd7-c78a8ecd6106": ("Lakehouse",    "weather_bronze_raw"),
        "27e1952e-cbd2-5a14-b54b-79f090db2745": ("KQLDatabase",  "bikerentaleventhouse"),
    }

    guid_map = {}  # logicalId → actual GUID

    # Look up Lakehouses
    resp = requests.get(f"{API}/workspaces/{ws_id}/lakehouses", headers=headers)
    resp.raise_for_status()
    for lh in resp.json().get("value", []):
        for lid, (ltype, lname) in LOGICAL_IDS.items():
            if ltype == "Lakehouse" and lh["displayName"] == lname:
                guid_map[lid] = lh["id"]

    # Look up KQL Databases
    resp = requests.get(f"{API}/workspaces/{ws_id}/kqlDatabases", headers=headers)
    resp.raise_for_status()
    for db in resp.json().get("value", []):
        for lid, (ltype, lname) in LOGICAL_IDS.items():
            if ltype == "KQLDatabase" and db["displayName"] == lname:
                guid_map[lid] = db["id"]

    print(f"   🔍 Resolved {len(guid_map)}/3 destination GUIDs")
    for lid, actual in guid_map.items():
        _, lname = LOGICAL_IDS[lid]
        print(f"      {lname}: {lid[:8]}… → {actual[:8]}…")
    sys.stdout.flush()

    if len(guid_map) < 3:
        missing = [LOGICAL_IDS[k][1] for k in LOGICAL_IDS if k not in guid_map]
        msg = f"Missing destination items: {missing}"
        print(f"   ⚠️ {msg}")
        print("   Eventstreams will be skipped — create manually in Fabric UI")
        es_error_detail = msg
        es_ok = False
    else:
        # Deploy each Eventstream via REST API
        for es_folder in item_index.get("Eventstream", []):
            es_name = es_folder.rsplit(".", 1)[0]
            es_json_path = os.path.join(LAKEHOUSE_PATH, es_folder, "eventstream.json")

            with open(es_json_path, "r", encoding="utf-8") as f:
                es_def = f.read()

            # Patch workspace GUID
            es_def = es_def.replace(SOURCE_WORKSPACE_ID, ws_id)

            # Patch destination logicalIds → actual item GUIDs
            for lid, actual_id in guid_map.items():
                es_def = es_def.replace(lid, actual_id)

            # Strip "schema": "dbo" — target Lakehouses may not have schema enabled
            es_obj = json.loads(es_def)
            for dest in es_obj.get("destinations", []):
                dest.get("properties", {}).pop("schema", None)
            es_def = json.dumps(es_obj, indent=2, ensure_ascii=False)

            # Base64-encode the patched definition
            es_b64 = base64.b64encode(es_def.encode("utf-8")).decode("utf-8")
            definition_body = {
                "parts": [{"path": "eventstream.json",
                           "payload": es_b64,
                           "payloadType": "InlineBase64"}]
            }

            # Check if Eventstream already exists
            resp = requests.get(f"{API}/workspaces/{ws_id}/eventstreams", headers=headers)
            resp.raise_for_status()
            existing_id = None
            for es in resp.json().get("value", []):
                if es["displayName"] == es_name:
                    existing_id = es["id"]
                    break

            if existing_id:
                print(f"   📝 Updating Eventstream '{es_name}' ({existing_id[:8]}…)")
                sys.stdout.flush()
                resp = requests.post(
                    f"{API}/workspaces/{ws_id}/eventstreams/{existing_id}/updateDefinition",
                    headers=headers, json={"definition": definition_body}
                )
            else:
                print(f"   📝 Creating Eventstream '{es_name}'…")
                sys.stdout.flush()
                resp = requests.post(
                    f"{API}/workspaces/{ws_id}/eventstreams",
                    headers=headers,
                    json={"displayName": es_name, "definition": definition_body}
                )

            if resp.status_code in (200, 201):
                try:
                    result_id = resp.json().get("id", "") if resp.text.strip() else ""
                except Exception:
                    result_id = ""
                print(f"   ✅ Eventstream '{es_name}' deployed" + (f" ({result_id[:8]}…)" if result_id else ""))
            elif resp.status_code == 202:
                result = wait_lro(resp, headers, f"Eventstream '{es_name}'", max_wait=120)
                if result:
                    print(f"   ✅ Eventstream '{es_name}' deployed")
                else:
                    msg = f"Eventstream '{es_name}' LRO failed"
                    print(f"   ❌ {msg}")
                    es_error_detail += msg + "\n"
                    es_ok = False
            else:
                msg = f"Eventstream '{es_name}' HTTP {resp.status_code}: {resp.text[:400]}"
                print(f"   ❌ {msg}")
                es_error_detail += msg + "\n"
                es_ok = False
            sys.stdout.flush()

except Exception as e:
    es_ok = False
    tb = traceback.format_exc()
    es_error_detail = f"Exception: {e}\n{tb}"
    print(f"   ❌ Eventstreams FAILED: {e}")
    print(f"   Traceback:\n{tb}")
    sys.stdout.flush()

if es_ok:
    print("   ✅ Eventstreams deployed via REST API")

# --- 5b: Pipeline + Dashboard + Agents via fabric-cicd ---
cicd_ok = True
try:
    stage_dir, stage_ws, _ = make_stage_dir(
        ["DataPipeline", "KQLDashboard"],
        ref_types=["KQLDatabase", "Lakehouse", "Eventhouse", "Eventstream", "Notebook", "SemanticModel"])
    deploy_types = ["DataPipeline", "KQLDashboard"]
    folders = []
    for t in deploy_types:
        folders.extend(item_index.get(t, []))
    print(f"   📦 Deploying: {', '.join(f.rsplit('.', 1)[0] for f in folders)}")

    ws = FabricWorkspace(workspace_id=ws_id, repository_directory=stage_ws,
                         item_type_in_scope=deploy_types)
    publish_all_items(ws)
    shutil.rmtree(stage_dir, ignore_errors=True)
    print("   ✅ Pipeline + Dashboard deployed via fabric-cicd")
except Exception as e:
    cicd_ok = False
    print(f"   ❌ Pipeline + Dashboard FAILED: {e}")

stage_status["5_Remaining"] = es_ok and cicd_ok
if stage_status["5_Remaining"]:
    print("   ✅ Stage 5/5 complete")
else:
    detail = []
    if not es_ok:
        detail.append("Eventstreams")
    if not cicd_ok:
        detail.append("Pipeline/Dashboard")
    print(f"   ❌ Stage 5/5 partial failure: {', '.join(detail)}")

# ═══════════════════════════════════════════════════════════════
# DEPLOYMENT SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
failed = [k for k, v in stage_status.items() if not v]
if failed:
    print(f"⚠️ DEPLOYMENT PARTIALLY COMPLETE — {len(failed)} stage(s) failed")
else:
    print("✅ DEPLOYMENT COMPLETE — 21 items deployed")
print(f"{'='*60}")
for stage_name, ok in stage_status.items():
    icon = "✅" if ok else "❌"
    print(f"   {icon} {stage_name.replace('_', ': ', 1)}")
# Re-print Eventstream errors at the VERY END so they're visible
# (Fabric notebook output truncates earlier lines when fabric-cicd logs are appended)
if es_error_detail:
    print(f"\n{'─'*60}")
    print("📋 EVENTSTREAM ERROR DETAILS:")
    print(f"{'─'*60}")
    print(es_error_detail.strip())
    print(f"{'─'*60}")

if failed:
    print(f"\nTo retry failed stages, re-run this cell — fabric-cicd is idempotent.")
print("\nNext: Run Cell 2.5 to upload sample_data, then Cell 3 to fix placeholders, then Cell 4 to validate.")



In [ ]:
# =============================================================
# CELL 2.5 — Upload sample_data to bicycles_gold Lakehouse
# =============================================================
# Copies sample_data files (dim_customers.csv etc.) from the staging
# lakehouse to the deployed bicycles_gold lakehouse.
#
# Prerequisites:
#   1. Upload sample_data/ folder to staging lakehouse Files/
#   2. Run Cell 2 first (deploys bicycles_gold lakehouse)

import os
import requests
import sempy.fabric as fabric

print("📦 UPLOAD SAMPLE_DATA TO BICYCLES_GOLD")
print("=" * 50)

ws_id = fabric.get_workspace_id()
token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
API = "https://api.fabric.microsoft.com/v1"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Find bicycles_gold lakehouse
items = requests.get(f"{API}/workspaces/{ws_id}/items", headers=headers).json().get("value", [])
gold_lh = None
for item in items:
    if item["displayName"] == "bicycles_gold" and item["type"] == "Lakehouse":
        gold_lh = item
        break

if not gold_lh:
    print("❌ bicycles_gold lakehouse not found! Run Cell 2 first.")
else:
    gold_lh_id = gold_lh["id"]
    print(f"  Found bicycles_gold: {gold_lh_id}")
    
    # Find sample_data in staging lakehouse
    staging_paths = [
        "/lakehouse/default/Files/sample_data",
        "/lakehouse/default/Files/RTI-Hackathon-Demo/sample_data",
        "/lakehouse/default/Files/RTI-Hackathon-Demo-main/sample_data",
    ]
    
    sample_data_path = None
    for path in staging_paths:
        if os.path.isdir(path):
            sample_data_path = path
            break
    
    if not sample_data_path:
        print("⚠️  sample_data folder not found in staging lakehouse!")
        print("   Upload sample_data/ folder to staging lakehouse Files/")
    else:
        print(f"  Found staging sample_data: {sample_data_path}")
        
        # Copy each CSV to bicycles_gold
        target_base = f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{gold_lh_id}/Files/sample_data"
        
        csv_files = [f for f in os.listdir(sample_data_path) if f.endswith(".csv")]
        if not csv_files:
            print("   No CSV files found in sample_data/")
        else:
            for csv_file in csv_files:
                src = f"file:{sample_data_path}/{csv_file}"  # file: prefix for local path
                dst = f"{target_base}/{csv_file}"
                print(f"  Copying {csv_file}...")
                try:
                    notebookutils.fs.cp(src, dst, recurse=False)
                    print(f"    ✅ {csv_file} → bicycles_gold/Files/sample_data/")
                except Exception as e:
                    print(f"    ❌ Failed: {e}")
        
        print(f"\n✅ Sample data uploaded to bicycles_gold")
        print(f"   Path: Files/sample_data/")

print("\nNext: Run Cell 3 to auto-fix placeholders.")

In [ ]:
# =============================================================
# CELL 3 — Auto-Fix Placeholders (KQL Dashboard + Pipeline)
# =============================================================
# After deployment, 2 placeholders remain that fabric-cicd can't resolve:
#   1. __EVENTHOUSE_QUERY_URI__  in the KQL Dashboard
#   2. SM refresh activity in the Pipeline (needs manual connection)
#
# This cell patches them automatically via the Fabric REST API.

import requests, json, base64, time
import sempy.fabric as fabric

ws_id = fabric.get_workspace_id()
token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
API = "https://api.fabric.microsoft.com/v1"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

def wait_for_lro(resp, label="operation"):
    """Poll long-running operation until complete."""
    if resp.status_code == 202:
        loc = resp.headers.get("Location") or resp.headers.get("x-ms-operation-url")
        if loc:
            for _ in range(60):
                time.sleep(3)
                poll = requests.get(loc, headers={"Authorization": f"Bearer {token}"})
                if poll.status_code == 200:
                    body = poll.json()
                    status = body.get("status", "").lower()
                    if status in ("succeeded", "completed"):
                        print(f"   ✅ {label} completed")
                        return True
                    elif status == "failed":
                        print(f"   ❌ {label} failed: {body}")
                        return False
            print(f"   ⚠️ {label} timed out")
            return False
    return None

# ── List workspace items ──
items = requests.get(f"{API}/workspaces/{ws_id}/items", headers=headers).json().get("value", [])
item_map = {i["displayName"]: i for i in items}

# ═══════════════════════════════════════════════════════════════
# FIX 1: KQL Dashboard — replace __EVENTHOUSE_QUERY_URI__ + __KQL_DB_ID__
# ═══════════════════════════════════════════════════════════════
print("🔧 Fix 1: KQL Dashboard — Eventhouse query URI + KQL DB scope")

eh = item_map.get("bikerentaleventhouse")
dash = None
for i in items:
    if i["type"] == "KQLDashboard":
        dash = i
        break

if eh and dash:
    # Get Eventhouse query URI — try multiple API endpoints and field names
    # The Eventhouse API may return the URI under different property names
    query_uri = ""

    # Method 1: GET /eventhouses/{id}
    eh_resp = requests.get(
        f"{API}/workspaces/{ws_id}/eventhouses/{eh['id']}", headers=headers
    )
    if eh_resp.status_code == 200:
        eh_detail = eh_resp.json()
        props = eh_detail.get("properties", {})
        query_uri = (props.get("queryServiceUri")
                     or props.get("queryUri")
                     or props.get("ingestionUri")
                     or props.get("parentEventhouseUri")
                     or "")
        if not query_uri:
            # Dump all properties keys for debugging
            print(f"   ℹ️  Eventhouse properties keys: {list(props.keys())}")
            print(f"   ℹ️  Full properties: {json.dumps(props, indent=2)[:500]}")
    else:
        print(f"   ℹ️  GET eventhouses/{eh['id']}: HTTP {eh_resp.status_code}")

    # Discover KQL Database ID (needed for dashboard scopeId)
    kql_db_id = ""
    db_resp = requests.get(f"{API}/workspaces/{ws_id}/kqlDatabases", headers=headers)
    if db_resp.status_code == 200:
        for db in db_resp.json().get("value", []):
            db_detail = requests.get(
                f"{API}/workspaces/{ws_id}/kqlDatabases/{db['id']}", headers=headers
            ).json()
            db_props = db_detail.get("properties", {})
            if not kql_db_id:
                kql_db_id = db["id"]
            uri = (db_props.get("queryUri")
                   or db_props.get("parentEventhouseUri")
                   or db_props.get("queryServiceUri")
                   or "")
            if uri and not query_uri:
                query_uri = uri
                print(f"   ℹ️  Got URI from KQL Database '{db['displayName']}': {uri}")
    if kql_db_id:
        print(f"   KQL Database ID: {kql_db_id}")

    if query_uri:
        print(f"   Eventhouse URI: {query_uri}")
        # Get dashboard definition
        resp = requests.post(
            f"{API}/workspaces/{ws_id}/items/{dash['id']}/getDefinition",
            headers=headers,
        )
        result = wait_for_lro(resp, "Get Dashboard Definition")

        if resp.status_code == 200:
            defn = resp.json()
        elif result:
            defn = requests.get(
                resp.headers.get("Location") or resp.headers.get("x-ms-operation-url"),
                headers={"Authorization": f"Bearer {token}"},
            ).json()
        else:
            defn = None

        if defn:
            parts = defn.get("definition", {}).get("parts", [])
            new_parts = []
            patched = False
            for part in parts:
                raw = base64.b64decode(part["payload"]).decode("utf-8")
                changed = False
                if "__EVENTHOUSE_QUERY_URI__" in raw:
                    raw = raw.replace("__EVENTHOUSE_QUERY_URI__", query_uri)
                    changed = True
                if "__KQL_DB_ID__" in raw and kql_db_id:
                    raw = raw.replace("__KQL_DB_ID__", kql_db_id)
                    changed = True
                # Also patch any leftover hardcoded source-workspace scopeId
                if "12bbefb2-a1e9-554c-a78e-873a7580aa10" in raw and kql_db_id:
                    raw = raw.replace("12bbefb2-a1e9-554c-a78e-873a7580aa10", kql_db_id)
                    changed = True
                if changed:
                    patched = True
                    part = dict(part)
                    part["payload"] = base64.b64encode(
                        raw.encode("utf-8")
                    ).decode("utf-8")
                new_parts.append(part)

            if patched:
                update_body = {"definition": {"parts": new_parts}}
                resp = requests.post(
                    f"{API}/workspaces/{ws_id}/items/{dash['id']}/updateDefinition",
                    headers=headers,
                    json=update_body,
                )
                result = wait_for_lro(resp, "Update Dashboard")
                if resp.status_code in (200, 201) or result:
                    print("   ✅ KQL Dashboard patched")
                else:
                    print(f"   ⚠️ Dashboard update returned HTTP {resp.status_code}")
                    print(f"      {resp.text[:300]}")
            else:
                print("   ℹ️  Dashboard already has correct URI and scopeId (no placeholders found)")
    else:
        print("   ⚠️ Could not get Eventhouse query URI from any source")
else:
    if not eh:
        print("   ⚠️ Eventhouse 'bikerentaleventhouse' not found")
    if not dash:
        print("   ⚠️ No KQL Dashboard found")

# ═══════════════════════════════════════════════════════════════
# FIX 2: Pipeline — remove SM refresh activity
# ═══════════════════════════════════════════════════════════════
print("\n🔧 Fix 2: Pipeline — remove SM refresh activity")

pipeline = item_map.get("PL_BicycleRTI_Medallion")
if pipeline:
    pipeline_id = pipeline["id"]
    resp = requests.post(
        f"{API}/workspaces/{ws_id}/items/{pipeline_id}/getDefinition",
        headers=headers,
    )
    result = wait_for_lro(resp, "Get Pipeline Definition")

    if resp.status_code == 200:
        defn = resp.json()
    elif result:
        defn = requests.get(
            resp.headers.get("Location") or resp.headers.get("x-ms-operation-url"),
            headers={"Authorization": f"Bearer {token}"},
        ).json()
    else:
        defn = None

    if defn:
        parts = defn.get("definition", {}).get("parts", [])
        new_parts = []
        patched = False
        for part in parts:
            raw = base64.b64decode(part["payload"]).decode("utf-8")
            if "__SM_REFRESH_CONN__" in raw or "Refresh Semantic Model" in raw:
                try:
                    pipeline_json = json.loads(raw)
                    activities = pipeline_json.get("properties", {}).get("activities", [])
                    orig_count = len(activities)
                    activities = [
                        a for a in activities
                        if a.get("name") != "Refresh Semantic Model"
                        and "__SM_REFRESH_CONN__" not in json.dumps(a)
                    ]
                    if len(activities) < orig_count:
                        pipeline_json["properties"]["activities"] = activities
                        # Also clean dependsOn references
                        for a in activities:
                            deps = a.get("dependsOn", [])
                            a["dependsOn"] = [
                                d for d in deps
                                if d.get("activity") != "Refresh Semantic Model"
                            ]
                        raw = json.dumps(pipeline_json)
                        patched = True
                except json.JSONDecodeError:
                    pass
                part = dict(part)
                part["payload"] = base64.b64encode(
                    raw.encode("utf-8")
                ).decode("utf-8")
            new_parts.append(part)

        if patched:
            update_body = {"definition": {"parts": new_parts}}
            resp = requests.post(
                f"{API}/workspaces/{ws_id}/items/{pipeline_id}/updateDefinition",
                headers=headers,
                json=update_body,
            )
            result = wait_for_lro(resp, "Update Pipeline")
            if resp.status_code in (200, 201) or result:
                print("   ✅ Pipeline patched — SM refresh removed (refresh manually after run)")
            else:
                print(f"   ⚠️ Pipeline update returned HTTP {resp.status_code}")
                print(f"      {resp.text[:300]}")
        else:
            print("   ℹ️  Pipeline already clean (no SM refresh activity)")
    else:
        print("   ⚠️ Could not retrieve pipeline definition")
else:
    print("   ⚠️ Pipeline 'PL_BicycleRTI_Medallion' not found")


# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("✅ AUTO-FIX COMPLETE")
print(f"{'='*60}")
print()
print("What was fixed:")
print("  • KQL Dashboard → clusterUri now points to your Eventhouse")
print("  • Pipeline → SM refresh activity removed (avoids connection error)")
print()
print("Remaining manual steps (IN THIS ORDER):")
print("  1. START EVENTSTREAMS — data must flow into Bronze before anything else")
print("     Open RTIbikeRental + RTI-WeatherDemo → confirm running (click Start if not)")
print("     Wait until you see data in Bronze lakehouses (10-30 min for meaningful volume)")
print("  2. Run the pipeline: PL_BicycleRTI_Medallion (Bronze → Silver → Gold → ML → Ontology)")
print("  3. Manually refresh both Semantic Models after pipeline completes")
print("  4. Run Post_Deploy_Setup.ipynb → creates Ontology + Graph + Agent")
print("  5. Graph Model → click 'Refresh now' (after ontology + pipeline data)")
print("  6. Verify KQL Dashboard shows data from Eventhouse")
print()
print("TIP: Pause eventstreams once you have enough data to save capacity.")


In [ ]:
# =============================================================
# CELL 4 — Validate Deployment
# =============================================================
import sempy.fabric as fabric

ws_id = fabric.get_workspace_id()
print(f"Workspace: {ws_id}")

# List all items
items = fabric.list_items()
print(f"\nTotal items: {len(items)}")

# Group by type
from collections import Counter
type_counts = Counter(row['Type'] for _, row in items.iterrows())
print("\nItems by type:")
for t, c in sorted(type_counts.items()):
    print(f"  {t}: {c}")

# ── Items deployed by THIS notebook (Cell 2) ──
deployed_here = [
    "bicycles_gold", "bikerental_bronze_raw", "bicycles_silver", "weather_bronze_raw",
    "bikerentaleventhouse",
    "RTIbikeRental", "RTI-WeatherDemo",
    "PL_BicycleRTI_Medallion",
    "Bicycle RTI Analytics", "Bicycle Ontology Model",
    "Bicycle Fleet Intelligence — Live Operations",
]
names = set(items['Display Name'])

print("\n── Deployed by this notebook (expected: all ✅) ──")
all_ok = True
for e in deployed_here:
    found = e in names
    status = "✅" if found else "❌ MISSING"
    if not found:
        all_ok = False
    print(f"  {status} {e}")

# ── Items deployed later by Post_Deploy_Setup.ipynb ──
post_deploy = [
    "Bicycle Fleet Intelligence Agent",
    "ontology data agent",
    "Bicycle_Ontology_Model_New",
    "Bicycle_Ontology_Model_New_graph",
    "Cycling-Campaign-Agent",
    "BicycleFleet_Activator",
    "Cycling Campaign Activator",
]

print("\n── Deployed later by Post_Deploy_Setup.ipynb ──")
for e in post_deploy:
    found = e in names
    status = "✅ already exists" if found else "⏳ not yet (expected)"
    print(f"  {status} {e}")

if all_ok:
    print("\n🎉 All items from this notebook deployed successfully!")
    print("   Next: Run Cell 5 to organize into folders, then Cell 6 for remaining manual steps.")
else:
    print("\n⚠️  Some items are missing — check the Cell 2 output for errors.")

In [ ]:
# =============================================================
# CELL 5 — Organize Workspace Items into Folders
# =============================================================
# Moves deployed items into folders matching the source workspace
# structure. Safe to re-run — items already in correct folders are skipped.
#
# Items not yet deployed (e.g. Post_Deploy items) are silently skipped.
# Re-run this cell after Post_Deploy_Setup.ipynb to organize those too.

import requests, json, time, re as _re
import sempy.fabric as fabric
from datetime import datetime, timezone

ws_id = fabric.get_workspace_id()
token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
API = "https://api.fabric.microsoft.com/v1"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
ITEMS_API = f"{API}/workspaces/{ws_id}/items"
FOLDERS_API = f"{API}/workspaces/{ws_id}/folders"

# ── Cleanup stale folders from previous runs ──
STALE_FOLDERS = ["Agent", "Operation Agent", "ML model"]

def cleanup_stale_folders():
    """Delete empty folders left by previous versions of this cell."""
    r = requests.get(FOLDERS_API, headers=headers)
    if r.status_code != 200:
        return
    for f in r.json().get("value", []):
        if f["displayName"] in STALE_FOLDERS:
            rd = requests.delete(f"{FOLDERS_API}/{f['id']}", headers=headers)
            if rd.status_code in (200, 204):
                print(f"  🗑️  Deleted stale folder: {f['displayName']}")
            elif rd.status_code == 400 and "not empty" in rd.text.lower():
                print(f"  ⚠️  Stale folder '{f['displayName']}' is not empty — move items out manually")

print("Cleaning up stale folders from previous runs...")
cleanup_stale_folders()

# ── Helper functions ──

def get_or_create_folder(name):
    """Get existing folder by name or create it. Returns folder ID."""
    r = requests.get(FOLDERS_API, headers=headers)
    if r.status_code == 200:
        for f in r.json().get("value", []):
            if f["displayName"] == name:
                return f["id"]
    r = requests.post(FOLDERS_API, headers=headers, json={"displayName": name})
    if r.status_code in (200, 201):
        return r.json()["id"]
    elif r.status_code == 409 or "AlreadyInUse" in r.text:
        r2 = requests.get(FOLDERS_API, headers=headers)
        for f in r2.json().get("value", []):
            if f["displayName"] == name:
                return f["id"]
    print(f"  [WARN] Could not create folder '{name}': {r.status_code}")
    return None

def move_item(item_id, item_name, folder_id, max_retries=3):
    """Move an item into a folder with retry on 429 rate limiting."""
    for attempt in range(max_retries + 1):
        r = requests.post(f"{ITEMS_API}/{item_id}/move",
                          headers=headers,
                          json={"targetFolderId": folder_id})
        if r.status_code == 200:
            return "moved"
        elif r.status_code == 400 and "already" in r.text.lower():
            return "moved"  # already in folder
        elif r.status_code == 429:
            retry_after = 10 * (2 ** attempt)
            try:
                match = _re.search(r'until:\s*([\d/]+\s+[\d:]+\s+\w+)', r.text)
                if match:
                    blocked_until = datetime.strptime(match.group(1), "%m/%d/%Y %I:%M:%S %p")
                    blocked_until = blocked_until.replace(tzinfo=timezone.utc)
                    wait_secs = max(1, (blocked_until - datetime.now(timezone.utc)).total_seconds() + 2)
                    retry_after = min(wait_secs, 120)
            except Exception:
                pass
            if attempt < max_retries:
                print(f"    ⏳ {item_name} — rate limited, retrying in {int(retry_after)}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(retry_after)
            else:
                print(f"    ⚠️  {item_name} — rate limited after {max_retries} retries")
                return "failed"
        elif r.status_code == 400 and "CannotMoveChildOnly" in r.text:
            return "child"  # KQLDatabase moves with Eventhouse parent
        else:
            print(f"    ⚠️  {item_name}: {r.status_code} {r.text[:150]}")
            return "failed"
    return "failed"

# ── Folder → item mapping ──
# Includes ALL items (Deploy + Post_Deploy). Items not yet deployed are skipped gracefully.
# For items where displayName collides (Eventhouse vs KQLDatabase), use (name, type) tuples.
FOLDER_MAP = {
    "Lakehouse": [
        "bikerental_bronze_raw",
        "weather_bronze_raw",
        "bicycles_silver",
        "bicycles_gold",
    ],
    "Eventhouse": [
        ("bikerentaleventhouse", "Eventhouse"),  # KQLDatabase child moves with parent
    ],
    "Notebooks": [
        "02_Bronze_Streaming_Ingest",
        "03_Silver_Enrich_Transform",
        "03a_Silver_Weather_Join",
        "04_Gold_Star_Schema",
        "05_KQL_Realtime_Queries",
        "06_ML_Demand_Forecast",
        "07_Activator_Alerts",
        "08_GeoAnalytics_HotSpots",
        "09_Ontology_Neighbourhood_Filter",
    ],
    "Semantic Models": [
        "Bicycle RTI Analytics",
        "Bicycle Ontology Model",
    ],
    "Pipeline": [
        "PL_BicycleRTI_Medallion",
    ],
    "RTI Dashboard": [
        "Bicycle Fleet Intelligence — Live Operations",
    ],
    "Real-Time Intelligence": [
        "RTIbikeRental",
        "RTI-WeatherDemo",
    ],
    "Ontology": [
        "Bicycle_Ontology_Model_New",
        "Bicycle_Ontology_Model_New_graph",
    ],
    "Agents": [
        "Bicycle Fleet Intelligence Agent",
        "ontology data agent",
        "Cycling-Campaign-Agent",
    ],
    "Activators": [
        "BicycleFleet_Activator",
        "Cycling Campaign Activator",
    ],
}

# ── Build item lookup with type disambiguation ──
print("\nRefreshing workspace items list...")
all_items_resp = requests.get(f"{ITEMS_API}?maxResults=500", headers=headers)
all_items = all_items_resp.json().get("value", []) if all_items_resp.status_code == 200 else []
print(f"  Found {len(all_items)} items in workspace\n")

# Primary lookup: name → id
item_lookup = {}
# Type-aware lookup: (name, type) → id
item_type_lookup = {}
for i in all_items:
    item_lookup[i["displayName"]] = i["id"]
    item_type_lookup[(i["displayName"], i["type"])] = i["id"]

# ── Execute moves ──
moved = 0
skipped = 0
not_found = 0

for folder_name, item_entries in FOLDER_MAP.items():
    folder_id = get_or_create_folder(folder_name)
    if not folder_id:
        continue
    print(f"📁 {folder_name} (ID: {folder_id[:8]}...)")

    for entry in item_entries:
        # Support both "name" and ("name", "Type") formats
        if isinstance(entry, tuple):
            name, item_type = entry
            item_id = item_type_lookup.get((name, item_type))
        else:
            name = entry
            item_id = item_lookup.get(name)

        if not item_id:
            print(f"    ⊘ {name} — not found (may not be deployed yet)")
            not_found += 1
            continue

        result = move_item(item_id, name, folder_id)
        if result == "moved":
            print(f"    ✅ {name}")
            moved += 1
        elif result == "child":
            print(f"    ↳ {name} — child item, moves with parent")
            moved += 1
        else:
            skipped += 1

        # Small delay between moves to avoid rate limiting
        time.sleep(1.5)

print(f"\n✅ Folder organization complete: {moved} moved, {skipped} failed, {not_found} not found")
if not_found > 0:
    print(f"   Items not found were likely not deployed yet — re-run after Post_Deploy_Setup.ipynb")
print(f"\nNext: See Cell 6 for remaining manual steps.")

## Remaining Manual Steps

Cells 3 + 5 auto-fixed placeholders and organized items into folders. These steps remain:

### 1. Start & Verify Eventstreams (Data Must Flow First)
Both eventstreams feed the **Bronze layer** — nothing downstream works without them.

1. Open each Eventstream and confirm it is **running** (they may auto-start):
   - **RTIbikeRental** — Bicycle sample data → `bikerental_bronze_raw` Lakehouse + `bikerentaleventhouse`
   - **RTI-WeatherDemo** — Real-time weather → `weather_bronze_raw` Lakehouse + `bikerentaleventhouse`
2. If not running, click **Start** on each one
3. **Wait until you see data arriving** — open the Bronze Lakehouse and refresh; you should see rows in the tables
4. You can let the streams run for 10-30 min to accumulate enough data for meaningful dashboards, or longer for richer analytics

> **Tip:** You can pause/stop the eventstreams once you have enough data to save capacity. You can always restart them later.

### 2. Run the Pipeline (Medallion Processing)
The pipeline processes Bronze → Silver → Gold → ML → Ontology. **It requires Bronze data from Step 1.**

1. Go to **PL_BicycleRTI_Medallion** pipeline
2. Click **Run** (~15-25 min)
3. After pipeline completes, **manually refresh** both Semantic Models:
   - Open each model → click **Refresh now**
   - (Cell 3 removed the auto-refresh activity because it needs a connection that can't be created programmatically)

> **Options:** You can run the pipeline on a schedule, trigger it once after enough streaming data, or run it immediately if you just want to validate the schema flows end-to-end.

### 3. Ontology + Graph Model (Post-Deploy Notebook)
1. Upload and run `Post_Deploy_Setup.ipynb` to create:
   - Bicycle_Ontology_Model_New (Ontology)
   - Bicycle_Ontology_Model_New_graph (GraphModel)
   - Cycling-Campaign-Agent (OperationsAgent)
2. After ontology is created, the Data Agent graph datasource will be wired automatically

### 4. Refresh Graph Model
1. Open the **Graph Model** → click **Refresh now**
2. This must be done after ontology creation AND after pipeline loads data

### 5. Verify KQL Dashboard
1. Open the **KQL Dashboard** — it pulls from the Eventhouse
2. Ensure the dashboard tiles show data (requires eventstream data from Step 1)
3. If tiles are empty, confirm the eventstreams ran long enough for data to arrive

### 6. Activator Rules (Optional)
See `docs/ACTIVATOR_SETUP.md` in the repo for rule definitions.
The Reflex/Activator items are not deployed automatically — create them manually if needed:
- **BicycleFleet_Activator**: Low stock alert, high demand surge
- **Cycling Campaign Activator**: Campaign trigger based on weather